# 01 — Data preprocessing pipeline

**Author:** Jai Sharma  
**Course:** MET CS 790 · Computer Vision in AI  
**Project:** Track 3 — Generative AI for HOA Disease Simulation  
**Date:** April 2026

---

## Purpose

This notebook processes raw OAI hand X-ray data into clean, split, training-ready datasets for our generative models (CycleGAN + Latent Diffusion).

## Input
- `data/raw/hand.xlsx` — 3,590 patients × 275 columns (KL, JSN, OP, ER scores)
- `data/raw/Finger_Joints.zip` — 41,060 cropped DIP/PIP/MCP joint X-ray images

## Output
- `data/splits/manifest_dip_only.csv` — 13,176 DIP images with KL labels + split assignment
- `data/splits/manifest.csv` — all joint types (DIP+PIP+MCP)
- `data/processed/merged_labels_YYYY-MM-DD.csv` — full label dump (107K rows)
- `data/processed/intensity_stats_YYYY-MM-DD.json` — normalization values (mean, std)
- `data/joint-samples/split_kl_samples_YYYY-MM-DD.png` — visual sanity check

## Key decisions
- Patient-level split (70/15/15) — no patient appears in multiple splits
- Stratified by max KL grade — rare KL 3/4 present in all splits
- Uses `id` column for image matching (not `duryeaid`)
- v00 labels assigned (images have no timepoint info in filenames)

In [1]:
# ============================================================================
# CELL 1: MOUNT DRIVE & SET PATHS
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# change this to match your Drive folder
PROJECT_ROOT = '/content/drive/MyDrive/CV-Project'

RAW_DIR       = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR    = os.path.join(PROJECT_ROOT, 'data', 'splits')
SAMPLES_DIR   = os.path.join(PROJECT_ROOT, 'data', 'joint-samples')

# local disk for fast image I/O
LOCAL_IMG_DIR = '/content/finger_joints_images'

# check raw files exist
assert os.path.exists(os.path.join(RAW_DIR, 'hand.xlsx')), 'hand.xlsx not found'
assert os.path.exists(os.path.join(RAW_DIR, 'Finger_Joints.zip')), 'Finger_Joints.zip not found'

for d in [PROCESSED_DIR, SPLITS_DIR, SAMPLES_DIR]:
    os.makedirs(d, exist_ok=True)

print('All paths verified.')

In [ ]:
# ============================================================================
# CELL 2: EXTRACT IMAGES TO LOCAL DISK
# ============================================================================
import zipfile
import time
import shutil

# always fresh extract
if os.path.exists(LOCAL_IMG_DIR):
    shutil.rmtree(LOCAL_IMG_DIR)

zip_path = os.path.join(RAW_DIR, 'Finger_Joints.zip')
print(f'Extracting {zip_path}...')
start = time.time()
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(LOCAL_IMG_DIR)
print(f'Done in {time.time() - start:.1f}s')

# count extracted files
all_files = []
for root, dirs, files in os.walk(LOCAL_IMG_DIR):
    for f in files:
        all_files.append(os.path.join(root, f))

print(f'Extracted {len(all_files)} files')

In [ ]:
# ============================================================================
# CELL 3: LOAD & EXPLORE hand.xlsx
# ============================================================================
import pandas as pd
import numpy as np

xlsx_path = os.path.join(RAW_DIR, 'hand.xlsx')
df_raw = pd.read_excel(xlsx_path)

print(f'Shape: {df_raw.shape}')
print(f'Columns ({len(df_raw.columns)}):')
for i, col in enumerate(df_raw.columns[:20]):
    print(f'  {i}: {col}')
print(f'  ... and {len(df_raw.columns)-20} more')

print(f'\nFirst 3 rows:')
df_raw.head(3)

In [ ]:
# ============================================================================
# CELL 4: MELT WIDE FORMAT → LONG FORMAT
# ============================================================================
import re

# the xlsx has columns like v00DIP2_KL, v06PIP3_OP, etc
# pattern: v{timepoint}{JointType}{Number}_{ScoreType}

id_col = 'duryeaid'
score_cols = [c for c in df_raw.columns if c != id_col]

records = []
for col in score_cols:
    m = re.match(r'^v(\d+)(DIP|PIP|MCP|IP|CMC|STT)(\d+)?_(\w+)$', col, re.IGNORECASE)
    if m:
        records.append({
            'column': col,
            'timepoint': f'v{m.group(1)}',
            'joint_type': m.group(2).upper(),
            'joint_num': m.group(3),
            'score_type': m.group(4),
        })

parsed_df = pd.DataFrame(records)
print(f'Parsed {len(records)} score columns')
print(f'Timepoints: {parsed_df["timepoint"].unique()}')
print(f'Joint types: {parsed_df["joint_type"].unique()}')
print(f'Score types: {parsed_df["score_type"].unique()}')

In [ ]:
# ============================================================================
# CELL 5: BUILD LONG-FORMAT DATAFRAME
# ============================================================================
# important: use 'id' column (OAI participant ID) not 'duryeaid'
# duryeaid is an internal scoring ID that doesn't match image filenames
# the 'id' column matches the patient IDs in image filenames

rows = []
for _, patient_row in df_raw.iterrows():
    patient_id = int(patient_row['id'])

    for col in score_cols:
        m = re.match(r'^v(\d+)(DIP|PIP|MCP|IP|CMC|STT)(\d+)?_(\w+)$', col, re.IGNORECASE)
        if not m:
            continue

        timepoint  = f'v{m.group(1)}'
        joint_type = m.group(2).upper()
        joint_num  = m.group(3)
        score_type = m.group(4)
        value = patient_row[col]

        joint_id = f'{joint_type}_{joint_num}' if joint_num else joint_type

        rows.append({
            'patient_id': patient_id,
            'timepoint': timepoint,
            'joint_type': joint_type,
            'joint_num': joint_num,
            'joint_id': joint_id,
            'score_type': score_type,
            'value': value,
        })

df_long = pd.DataFrame(rows)
print(f'Long-format shape: {df_long.shape}')
df_long.head(10)

In [ ]:
# ============================================================================
# CELL 6: PIVOT → ONE ROW PER JOINT
# ============================================================================
df_pivot = df_long.pivot_table(
    index=['patient_id', 'timepoint', 'joint_type', 'joint_num', 'joint_id'],
    columns='score_type',
    values='value',
    aggfunc='first'
).reset_index()
df_pivot.columns.name = None

print(f'Pivoted shape: {df_pivot.shape}')
print(f'\nKL distribution (all joints):')
print(df_pivot['KL'].value_counts().sort_index())
print(f'\nJoint type distribution:')
print(df_pivot['joint_type'].value_counts())

In [ ]:
# ============================================================================
# CELL 7: MATCH IMAGES TO LABELS
# ============================================================================
import glob

# find all image files
image_extensions = ('*.png', '*.jpg', '*.jpeg', '*.tif', '*.tiff', '*.bmp')
image_files = []
for ext in image_extensions:
    image_files.extend(glob.glob(os.path.join(LOCAL_IMG_DIR, '**', ext), recursive=True))
print(f'Found {len(image_files)} image files')

# build lookup: parse filename {patient_id}_{joint_type}{joint_num}.png
image_lookup = {}
for fp in image_files:
    fname = os.path.basename(fp)
    m = re.match(r'^(\d+)_([a-zA-Z]+)(\d+)\.\w+$', fname)
    if m:
        pid = m.group(1)
        jtype = m.group(2).upper()
        jnum = m.group(3)
        key = f'{pid}_{jtype}_{jnum}'
        image_lookup[key] = fp

print(f'Built lookup with {len(image_lookup)} entries')

# match images — no timepoint info in filenames so we use v00 labels
def find_image(row):
    pid = str(int(row['patient_id']))
    jtype = row['joint_type']
    jnum = str(row['joint_num'])
    key = f'{pid}_{jtype}_{jnum}'
    return image_lookup.get(key, None)

df_pivot['image_path'] = df_pivot.apply(find_image, axis=1)

# filter to v00 only (images are from one timepoint, assumed baseline)
df_v00 = df_pivot[(df_pivot['timepoint'] == 'v00') & (df_pivot['image_path'].notna())].copy()

matched = len(df_v00)
print(f'\nMatched {matched} joints to images (v00 labels)')
print(f'\nBy joint type:')
print(df_v00['joint_type'].value_counts())

In [ ]:
# ============================================================================
# CELL 8: CHECK FOR CORRUPTED IMAGES
# ============================================================================
from PIL import Image
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

df_valid = df_v00.copy()
corrupted = []
sizes = []

print('Checking image integrity...')
for idx, row in df_valid.iterrows():
    fp = row['image_path']
    fsize = os.path.getsize(fp)
    if fsize < 1024:
        corrupted.append(idx)
        continue
    try:
        with Image.open(fp) as img:
            w, h = img.size
            sizes.append({'width': w, 'height': h})
    except:
        corrupted.append(idx)

df_valid = df_valid[~df_valid.index.isin(corrupted)].copy()
print(f'Valid: {len(df_valid)} | Corrupted: {len(corrupted)}')

if sizes:
    sizes_df = pd.DataFrame(sizes)
    print(f'Dimensions — W: {sizes_df["width"].median():.0f}, H: {sizes_df["height"].median():.0f}')

In [ ]:
# ============================================================================
# CELL 9: FILTER TO DIP JOINTS ONLY
# ============================================================================
df_dip = df_valid[df_valid['joint_type'] == 'DIP'].copy()

print(f'DIP joints: {len(df_dip)} images')
print(f'\nKL distribution (DIP):')
print(df_dip['KL'].value_counts().sort_index())

In [ ]:
# ============================================================================
# CELL 10: PATIENT-LEVEL TRAIN/VAL/TEST SPLIT (70/15/15)
# ============================================================================
from sklearn.model_selection import train_test_split

# get max KL per patient for stratification
patient_max_kl = df_dip.groupby('patient_id')['KL'].max().reset_index()
patient_max_kl.columns = ['patient_id', 'max_kl']
patient_max_kl['max_kl'] = patient_max_kl['max_kl'].fillna(-1).astype(int)

# bin rare grades for stratification
kl_counts = patient_max_kl['max_kl'].value_counts()
patient_max_kl['strat_kl'] = patient_max_kl['max_kl'].copy()
rare = kl_counts[kl_counts < 5].index.tolist()
if rare:
    patient_max_kl.loc[patient_max_kl['strat_kl'].isin(rare), 'strat_kl'] = 99

# split 70/30 then 30 → 15/15
p_train, p_temp = train_test_split(
    patient_max_kl['patient_id'], test_size=0.30, random_state=42,
    stratify=patient_max_kl['strat_kl'])

temp_kl = patient_max_kl[patient_max_kl['patient_id'].isin(p_temp)]
p_val, p_test = train_test_split(
    temp_kl['patient_id'], test_size=0.50, random_state=42,
    stratify=temp_kl['strat_kl'])

train_set = set(p_train.values)
val_set = set(p_val.values)

def get_split(pid):
    if pid in train_set: return 'train'
    elif pid in val_set: return 'val'
    else: return 'test'

df_dip['split'] = df_dip['patient_id'].apply(get_split)
df_valid['split'] = df_valid['patient_id'].apply(get_split)

print(f'Patient split: Train={len(p_train)}, Val={len(p_val)}, Test={len(p_test)}')
print(f'\nImage split (DIP):')
print(df_dip['split'].value_counts())
print(f'\nKL per split:')
print(df_dip.groupby('split')['KL'].value_counts().unstack(fill_value=0))

In [ ]:
# ============================================================================
# CELL 11: DATA LEAKAGE CHECK
# ============================================================================
train_p = set(df_dip[df_dip['split']=='train']['patient_id'])
val_p   = set(df_dip[df_dip['split']=='val']['patient_id'])
test_p  = set(df_dip[df_dip['split']=='test']['patient_id'])

leak = (train_p & val_p) | (train_p & test_p) | (val_p & test_p)
if leak:
    print(f'DATA LEAKAGE: {len(leak)} patients in multiple splits!')
else:
    print('No data leakage — no patient appears in multiple splits.')

In [ ]:
# ============================================================================
# CELL 12: SAVE MANIFESTS
# ============================================================================
from datetime import date
today = date.today().isoformat()

# full manifest (DIP + PIP + MCP)
manifest_all_path = os.path.join(SPLITS_DIR, 'manifest.csv')
df_valid.to_csv(manifest_all_path, index=False)
print(f'Saved: manifest.csv ({len(df_valid)} rows)')

# DIP-only manifest (primary training file)
manifest_dip_path = os.path.join(SPLITS_DIR, 'manifest_dip_only.csv')
df_dip.to_csv(manifest_dip_path, index=False)
print(f'Saved: manifest_dip_only.csv ({len(df_dip)} rows)')

# merged labels to processed/
merged_path = os.path.join(PROCESSED_DIR, f'merged_labels_{today}.csv')
df_pivot.to_csv(merged_path, index=False)
print(f'Saved: merged_labels_{today}.csv ({len(df_pivot)} rows)')

In [ ]:
# ============================================================================
# CELL 13: VISUALIZE SAMPLES BY SPLIT × KL GRADE
# ============================================================================
import matplotlib.pyplot as plt

kl_grades = sorted(df_dip['KL'].dropna().unique())
n_kl = len(kl_grades)

fig, axes = plt.subplots(3, n_kl, figsize=(4*n_kl, 12))
for i, split in enumerate(['train', 'val', 'test']):
    for j, kl in enumerate(kl_grades):
        ax = axes[i][j]
        subset = df_dip[(df_dip['split']==split) & (df_dip['KL']==kl)]
        if len(subset) > 0:
            sample = subset.sample(1, random_state=42).iloc[0]
            try:
                img = Image.open(sample['image_path'])
                ax.imshow(img, cmap='gray')
                ax.set_title(f'{split} | KL {int(kl)}\n{sample["joint_id"]}', fontsize=10)
            except:
                ax.text(0.5, 0.5, 'Load error', ha='center', va='center')
        else:
            ax.text(0.5, 0.5, f'No KL {int(kl)}\nin {split}', ha='center', va='center')
        ax.axis('off')

plt.suptitle('Sample DIP Joint X-rays by Split and KL Grade', fontsize=14, fontweight='bold')
plt.tight_layout()
vis_path = os.path.join(SAMPLES_DIR, f'split_kl_samples_{today}.png')
plt.savefig(vis_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {vis_path}')

In [ ]:
# ============================================================================
# CELL 14: INTENSITY NORMALIZATION STATS
# ============================================================================
import json

print('Computing intensity statistics from training images...')
train_images = df_dip[df_dip['split']=='train']['image_path'].tolist()

sample_size = min(2000, len(train_images))
np.random.seed(42)
sampled = np.random.choice(train_images, sample_size, replace=False)

pixel_sum = 0
pixel_sq_sum = 0
pixel_count = 0

for fp in sampled:
    try:
        img = np.array(Image.open(fp).convert('L'), dtype=np.float32) / 255.0
        pixel_sum += img.sum()
        pixel_sq_sum += (img ** 2).sum()
        pixel_count += img.size
    except:
        continue

mean = pixel_sum / pixel_count
std = np.sqrt(pixel_sq_sum / pixel_count - mean ** 2)

print(f'\nMean: {mean:.4f}')
print(f'Std:  {std:.4f}')
print(f'Computed from {sample_size} training images')
print(f'\nUse: transforms.Normalize(mean=[{mean:.4f}], std=[{std:.4f}])')

stats = {'mean': float(mean), 'std': float(std),
         'n_images_sampled': sample_size, 'computed_on': today}
stats_path = os.path.join(PROCESSED_DIR, f'intensity_stats_{today}.json')
with open(stats_path, 'w') as f:
    json.dump(stats, f, indent=2)
print(f'Saved: {stats_path}')

In [ ]:
# ============================================================================
# CELL 15: FINAL SUMMARY
# ============================================================================
print('=' * 60)
print('  DATA PREPROCESSING PIPELINE — FINAL SUMMARY')
print('=' * 60)

print(f'\nRaw data:')
print(f'  {len(df_raw)} patients in hand.xlsx')
print(f'  {len(image_files)} images in Finger_Joints.zip')

print(f'\nProcessed:')
print(f'  {len(df_valid)} matched images (DIP+PIP+MCP)')
print(f'  {len(df_dip)} DIP joints (primary training set)')

print(f'\nSplit:')
for split in ['train', 'val', 'test']:
    n_p = df_dip[df_dip['split']==split]['patient_id'].nunique()
    n_i = len(df_dip[df_dip['split']==split])
    print(f'  {split:5s}: {n_p:4d} patients, {n_i:5d} images')

print(f'\nKL distribution (DIP):')
for kl in sorted(df_dip['KL'].dropna().unique()):
    n = len(df_dip[df_dip['KL']==kl])
    pct = n / len(df_dip) * 100
    print(f'  KL {int(kl)}: {n:5d} ({pct:.1f}%)')

print(f'\nNormalization: mean={mean:.4f}, std={std:.4f}')
print(f'Leakage check: PASSED')
print(f'\nDone!')